[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-ORG/YOUR-REPO/blob/main/tutorial_2_ml_pipeline.ipynb)

# Health Innovation Lab (HIL) — Tutorial 2
## Machine Learning with scikit-learn

Tutors: *(add your names/emails here)*

*Condensed and adapted for a 2-hour session from the original "Machine Learning with scikit-learn" tutorial, originally written by Jaime Rodríguez-Guerra and Jan Philipp Albrecht — itself based on prior work by Jan Philipp Albrecht and Jacob Gora ([TeachOpenCADD T7](https://github.com/volkamerlab/TeachOpenCADD/blob/master/talktorials/7_machine_learning/T7_machine_learning.ipynb)).*

Last session we cleaned and explored the Alzheimer's disease dataset. Today we go end to end: **prepare data → train → evaluate → interpret** — the same shape every ML project has.

<font color=green>Focus on *why* each step exists and what the numbers mean, not on writing every line yourself. Ask questions whenever something is unclear!</font>

***

## 1. Aims of this session

Familiarize yourself with the ground concepts of machine learning while you apply popular algorithms and patterns in Python's `scikit-learn`, on the same real dataset from last session.

## 2. Learning goals

### Theory
- Machine Learning (ML) methods
- Data preparation

### Practical
- Prepare your data
- Use regression to find correlations between MMSE and hippocampus volume
- Apply Random Forests to predict Alzheimer's disease based on volumetric measurements and other variables
- Cluster with k-means to guess the thresholds used for MMSE-based diagnosis
- Validate your models

## 3. References
- ML:
  - Random forest (RF): http://ect.bell-labs.com/who/tkh/publications/papers/odt.pdf
  - Support vector machines (SVM): https://link.springer.com/article/10.1007%2FBF00994018
  - Artificial neural networks (ANN): https://www.frontiersin.org/research-topics/4817/artificial-neural-networks-as-models-of-neural-information-processing
- Performance:
  - Sensitivity and specificity (Wikipedia): https://en.wikipedia.org/wiki/Sensitivity_and_specificity
  - ROC curve and AUC (Wikipedia): https://en.wikipedia.org/wiki/Receiver_operating_characteristic#Area_under_the_curve
- See also [this notebook by B. Merget](https://github.com/Team-SKI/Publications/tree/master/Profiling_prediction_of_kinase_inhibitors) from *J. Med. Chem.*, 2017, 60, 474−485

***

## 4. Theory

You heard about this already in today's lecture and the online course, so here just a little summary.

### Supervised vs unsupervised learning

In supervised learning, the algorithm is fed data that is well labelled (e.g. classification); e.g. some of the data will be tagged with/as the correct answer. In unsupervised learning, the algorithm infers the labels (e.g. clustering).

### Machine Learning (ML)

ML can be applied for (see also the [scikit-learn page](http://scikit-learn.org/stable/)):
- **Classification (supervised)**: Identify to which category an object belongs (Nearest neighbors, Naive Bayes, RF, SVM, ...)
- **Regression**: Prediction of a continuous-valued attribute associated with an object
- **Clustering (unsupervised)**: Automated grouping of similar objects into sets

**Supervised learning**

Learning algorithm creates rules by finding patterns in the training data.
- **Decision trees**: A decision tree is a flowchart-like structure in which each internal node represents a "test" on an attribute, each branch represents the outcome of the test, and each leaf node represents a class label (decision taken after computing all attributes). The paths from root to leaf represent classification rules.
- **Random Forest (RF)**: Multiple decision trees which produce a mean prediction.
- **Support Vector Machines (SVM)**: SVMs can efficiently perform a non-linear classification using what is called the kernel trick, implicitly mapping their inputs into high-dimensional feature spaces. A classifier based on the idea of maximizing the margin as objective function.

**Validation strategies**

All models must be validated before using them!

*Train-test split*

The data is split *beforehand* in two sets:
- The training set, which will be used to *train* the model.
- The testing set, which will be used to validate the answer provided by the trained model. This data shouldn't have been *seen* by the model.

There are several criteria to split the data into these two sets, but a common one is a random split where 20% of the data goes to the testing set and the remaining 80% goes to the training set. This is called an 80/20 split.

*K-fold cross validation*

This model validation technique splits the dataset in two groups in an iterative manner:
- Training data set: considered as the known dataset on which the model is trained
- Test dataset: unknown dataset on which the model is then tested
- Process is repeated k times

The goal is to test the ability of the model to predict data which it has never seen before, in order to flag problems known as over-fitting and to assess the generalization ability of the model.

**Performance measures**

Having defined a True Positive (TP), True Negative (TN), False Positive (FP), and False Negative (FN), we have:
- **Sensitivity**, also true positive rate: TPR = TP / (FN + TP)
- **Specificity**, also true negative rate: TNR = TN / (FP + TN)
- **Accuracy**, also the trueness: ACC = (TP + TN) / (TP + TN + FP + FN)
- **ROC-curve**, receiver operating characteristic curve: a graphical plot that illustrates the diagnostic ability of our classifier, plotting the sensitivity against the specificity
- **AUC**, the area under the ROC curve: describes the probability that a classifier will rank a randomly chosen positive instance higher than a negative one. Values between 0 and 1, the higher the better.

### Data preparation

Some algorithms will expect the data in a specific form, so before models are trained and evaluated, the data needs to be formatted and sanitized accordingly. Perhaps surprisingly, this is often one of the trickiest parts in ML: getting good data and preparing it for the study!

Some common tasks include:
- Dropping non-numeric values
- Cleaning unneeded columns
- Standardizing labels and/or magnitudes
- Normalizing ranges
- Converting measurement units
- Finding a good representation

***

## 5. Preparing the data

We reload the cleaned file from last session (feel free to rerun session 1's notebook first if you want fresh output).

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not os.path.exists("data"):
    # This notebook expects a `data/` folder (from the course repo) next to it.
    # Colab starts from a blank machine, so we grab just that folder from GitHub.
    !git clone --depth 1 https://github.com/ritterlab/health_innovation_lab.git _course_repo
    !cp -r _course_repo/data ./data
    print("Data copied from the course repo.")
elif not os.path.exists("data"):
    print("Not running in Colab and no local ./data folder found -- "
          "make sure you're running this from inside the course repo.")


In [ ]:
import pandas as pd

df = pd.read_csv("data/alzheimers_disease_reduced.csv")
df = df[df["VISCODE"] == "bl"]   # safety re-check, even though it should already be baseline-only

print(df.shape)
df.head()

In `scikit-learn`'s naming convention:
- `X` (uppercase) = the known data / features (e.g. MMSE score, hippocampus volume)
- `y` (lowercase) = the label you want to predict (e.g. diagnosis)

### 5.1 Emergency data cleanup

**Never trust your data, even after cleaning it once.** `DX` is supposed to only contain three labels (`CN`, `MCI`, `Dementia`) — let's check.

In [ ]:
df['DX'].unique()

If you see extra/stray values here, drop them — they're artifacts, not real classes:

In [ ]:
df = df.query("DX in ('CN', 'Dementia', 'MCI')").copy()
df['DX'].unique()

***
## 6. Regression: does hippocampus volume relate to MMSE score?

MMSE tests are used in diagnosing Alzheimer's; low scores often correlate with cognitive impairment. Some literature suggests reduced hippocampus volume correlates with dementia. Let's check with linear regression.

In [ ]:
regr_Xy = df[['MMSE', 'Hippocampus']].dropna()
print(regr_Xy.shape)
regr_Xy.describe()

In [ ]:
from sklearn.model_selection import train_test_split

regr_X = regr_Xy['MMSE']
regr_y = regr_Xy['Hippocampus']
regr_X_train, regr_X_test, regr_y_train, regr_y_test = train_test_split(
    regr_X, regr_y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()
regressor.fit(regr_X_train, regr_y_train)

Oops, we got an error! Fortunately, it is a very informative one that tells us how to fix it. The `LinearRegression` object supports several features per data point (several dimensions), so as a generalization the *shape* of the array will be two-dimensional. We can fix that following the advice in the error message — `.values.reshape(-1, 1)` turns our 1D column into a "many rows, 1 column" array (tip: `.values` is the numpy array sitting behind the DataFrame).

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()
regr_X_train_fixed = regr_X_train.values.reshape(-1, 1)
regressor.fit(regr_X_train_fixed, regr_y_train.values)

> **Exercise**
> What are the coefficients of the equation? Inspect the `regressor` object and print the fitted line's equation, using `regressor.coef_` (the slope) and `regressor.intercept_`.

In [ ]:
# your code here


<details>
<summary>Solution</summary>

```python
print(f'The fitted line is y = {regressor.coef_[0]:.3f}*x + {regressor.intercept_:.3f}')
```
</details>

Now we can *predict* values! You could solve the equation by hand, but there's a convenient method on the `regressor` itself.

In [ ]:
regr_y_predicted = regressor.predict(regr_X_test.values.reshape(-1, 1))
regr_y_predicted

How do the *predicted* values compare to the *actual* values? Let's measure some stats.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
regr_mse = mean_squared_error(regr_y_test, regr_y_predicted)
regr_r2 = r2_score(regr_y_test, regr_y_predicted)
print("MSE:", regr_mse, "; R2:", regr_r2)

An R² close to 0 means the line barely explains any of the variation in hippocampus volume — let's see why, visually.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.scatter(regr_X_test, regr_y_test, c="grey", marker='.')
ax.plot(regr_X_test, regr_y_predicted, c="blue")
ax.set_xlabel("MMSE")
ax.set_ylabel("Hippocampus")
ax.set_title(f"y = {regressor.coef_[0]:.1f}x + {regressor.intercept_:.1f}")
plt.show()

> **Interpretation exercise**
>
> The data is too spread out for a strong linear fit — you can maybe see a *trend*, but it's not very predictive. Why might a single variable (MMSE) not be enough to predict something as complex as hippocampus volume? What would you want to add?

*Your answer:*

> ...


***
## 7. Classification: predicting diagnosis from brain volume measurements

Literature suggests several volumetric brain measurements correlate with Alzheimer's progression: `Ventricles`, `Hippocampus`, `Entorhinal` cortex, `Fusiform` gyrus. Can a model use these to predict `DX`?

In [ ]:
rf_Xy = df[['Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'DX']].dropna()
rf_X = rf_Xy[['Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform']]
rf_y = rf_Xy['DX']

rf_X_train, rf_X_test, rf_y_train, rf_y_test = train_test_split(
    rf_X, rf_y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(rf_X_train, rf_y_train)

A **Random Forest** is many decision trees trained on slightly different subsets of the data, whose predictions get averaged/voted — this tends to be more robust than any single tree (and much less prone to the kind of overfitting a single deep tree shows).

In [ ]:
rf_y_predicted = rf_classifier.predict(rf_X_test)

print("Predicted:", list(rf_y_predicted[:10]))
print("Actual:   ", list(rf_y_test[:10].values))

### 7.1 How good is it? A first, simple measure

MSE and R² are for regression — for classification we want a **success rate**: the fraction of predictions that matched reality.

In [ ]:
rf_ok = (rf_y_test == rf_y_predicted).sum()
rf_total = rf_y_test.shape[0]
rf_rate = 100 * rf_ok / rf_total
print(f"RF predicted {rf_ok} out of {rf_total} correctly ({rf_rate:.1f}%).")

# scikit-learn has a shortcut for exactly this:
print(f"Same number via .score(): {100 * rf_classifier.score(rf_X_test, rf_y_test):.1f}%")

> **Exercise**
> `RandomForestClassifier` has many hyperparameters — e.g. `n_estimators` (number of trees). Try increasing it (e.g. to 500) and see if the score changes noticeably. What's the cost of a larger forest?

In [ ]:
# your code here
rf_classifier_custom = RandomForestClassifier(n_estimators=500, random_state=42)
rf_classifier_custom.fit(rf_X_train, rf_y_train)
print(f"Score = {100 * rf_classifier_custom.score(rf_X_test, rf_y_test):.1f}%")

### 7.2 A confusion matrix

A raw success rate hides *what kind* of mistakes the model makes. A confusion matrix breaks it down: rows = true label, columns = predicted label.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(rf_classifier, rf_X_test, rf_y_test, normalize='all')
plt.show()

> **Interpretation exercise**
>
> Which diagnosis class does the model confuse most often with another? Given that `MCI` (mild cognitive impairment) sits clinically *between* `CN` and `Dementia`, does that confusion make intuitive sense?

*Your answer:*

> ...


### 7.3 Is this actually reliable? Cross-validation

If you re-ran the train/test split above with a different random seed, you might get a noticeably different score — small test sets are noisy. **Cross-validation** repeats the split several times and reports the *spread* of scores, not just one lucky (or unlucky) number.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_classifier, rf_X, rf_y, cv=10)
print(f"Avg accuracy: {scores.mean():.2f} (+/- {2*scores.std():.2f})")

> **Interpretation exercise**
>
> Compare this average to the single train/test score from section 7.1. Are they similar or different? Given how wide the `+/-` range is, how confident should we be that this model is genuinely better than guessing the most common class every time?

*Your answer:*

> ...


***
## 8. Optional bonus: does MMSE alone do just as well?

Before trusting the volumetric-measurement model above, it's worth asking: is there a *much simpler* variable that would do just as well? MMSE *should* be a good predictor of diagnosis on its own — it's literally a diagnostic questionnaire.

In [ ]:
mmse_Xy = df[['MMSE', 'DX']].dropna()
mmse_X = mmse_Xy['MMSE'].values.reshape(-1, 1)
mmse_y = mmse_Xy['DX'].values

mmse_X_train, mmse_X_test, mmse_y_train, mmse_y_test = train_test_split(
    mmse_X, mmse_y, test_size=0.2, random_state=42
)

mmse_classifier = RandomForestClassifier(random_state=42)
mmse_classifier.fit(mmse_X_train, mmse_y_train)
print(f"Score using MMSE alone = {100 * mmse_classifier.score(mmse_X_test, mmse_y_test):.1f}%")

> **Interpretation exercise**
>
> How does this compare to the 4-feature volumetric model? If a single existing questionnaire score does (roughly) as well as four MRI-derived measurements, what does that tell you about the added value of the more complex, more expensive model?

*Your answer:*

> ...


***
## 9. Optional bonus: unsupervised learning — can clustering rediscover the diagnosis?

Everything above was **supervised**: we always had the correct `DX` label. What if we hid it and just asked an algorithm to find groups in the data on its own — the same idea as the sepsis-phenotyping example from the lecture?

We'll reduce our features to 2 dimensions with **PCA** (Principal Component Analysis, so we can actually plot them), then cluster with **k-means**.

In [ ]:
kmeans_Xy = df[['MMSE', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform', 'DX']].dropna()
kmeans_X = kmeans_Xy[['MMSE', 'Ventricles', 'Hippocampus', 'Entorhinal', 'Fusiform']]
kmeans_y = kmeans_Xy['DX']

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
reduced_X = pca.fit_transform(kmeans_X)

fig, ax = plt.subplots()
ax.scatter(reduced_X[:, 0], reduced_X[:, 1], marker='.')
ax.set_title("Data reduced to 2 dimensions (PCA), no labels used")
plt.show()

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=kmeans_y.nunique(), random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(reduced_X)

fig, ax = plt.subplots()
colormap = {label: color for label, color in zip(kmeans_y.unique(), 'rgb')}
colors = [colormap[label] for label in kmeans_y]
ax.scatter(reduced_X[:, 0], reduced_X[:, 1], marker='.', c=colors)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           marker='X', s=200, color='black', zorder=10)
ax.set_title("Same points, colored by TRUE diagnosis (crosses = cluster centers)")
print(colormap)
plt.show()

> **Interpretation exercise**
>
> Do the true-diagnosis colors line up with visually separate clusters, or is it mostly one big overlapping cloud? If clustering *can't* easily recover the labels here, what does that tell you — about these particular features, or about how separable these clinical groups actually are?

*Your answer:*

> ...


***
## 10. Discussion

Today we built a complete ML pipeline on real clinical data:

- Prepared data for `scikit-learn` and caught a data artifact *again* (never trust your data, even the second time!)
- Tried linear regression (MMSE → Hippocampus) and saw a genuinely weak relationship, honestly reported via R²
- Trained a Random Forest classifier on volumetric brain measurements to predict diagnosis
- Evaluated it properly: success rate, confusion matrix, and — crucially — cross-validation, which is what actually tells you whether a result is trustworthy
- Compared it to a much simpler baseline (MMSE alone) and to a label-free clustering approach

The biggest lesson isn't really about Random Forests or PCA — it's the discipline: **any model can look good on one split of the data. Evaluating and validating it honestly is what turns a number into evidence.** That discipline is exactly what's at stake in the ethics session that follows.

## 11. Quiz

1. Can you assume data is ever "ready" for modeling as-is? What preprocessing did we do — and would you call that a minor chore or a key pipeline step?
2. Any model can be made to look better by tuning hyperparameters on the test set. Why is that a problem, and what technique from today helps guard against it?
3. Why did we compare the Random Forest to a single-variable (MMSE) baseline? What would you conclude if the baseline had scored just as well?

***